# Question 1:

In [2]:
import io, requests
import pandas as pd
import numpy as np

url = "https://www.iposcoop.com/ipos-recently-filed/"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
df = pd.read_html(io.StringIO(resp.text))[0]

print(df.columns.tolist())  # check exact column names first
df = df[df["Expected To Trade"] == "Withdrawn"].copy()
print(len(df))  # should be around 32

['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']
35


In [3]:
def classify(name):
    if "Technologies" in name: return "Technologies"
    if "Acquisition Corp" in name or "Corp" in name: return "Acquisition Corp"
    if "Inc" in name or "Incorporated" in name: return "Inc."
    if "Group" in name: return "Group"
    if "Ltd" in name or "Limited" in name: return "Limited"
    if "Holdings" in name or "Holding" in name: return "Holdings"
    return "Other"

df["Company Type"] = df["Company"].apply(classify)  # use your actual name column

In [4]:
def to_num(x):
    return pd.to_numeric(str(x).replace("$", "").replace(",", ""), errors="coerce")

for c in ["Price Low", "Price High", "Shares (millions)", "Est $ Vol (millions)"]:
    df[c] = df[c].apply(to_num)

df["Avg_price"] = (df["Price Low"] + df["Price High"]) / 2
df["Shares_offered_value"] = (
    (df["Shares (millions)"] * df["Avg_price"])
    .fillna(df["Est $ Vol (millions)"])
)

In [5]:
df.groupby("Company Type")["Shares_offered_value"].sum().sort_values(ascending=False)

,Shares_offered_value
Company Type,
Other,1115.4450
Acquisition Corp,499.9850
Inc.,351.0000
Holdings,311.6575
Limited,219.2500
Technologies,184.9000
Group,56.5750


In [6]:
df["File Date"] = pd.to_datetime(df["File Date"], errors="coerce")
print(df["File Date"].min(), df["File Date"].max())

df = df[df["File Date"] < "2026-09-11"]
print(len(df))

print(df[df["Company Type"] == "Other"]
      [["File Date", "Company", "Shares_offered_value"]]
      .sort_values("Shares_offered_value", ascending=False))

print(df.groupby("Company Type")["Shares_offered_value"].sum().sort_values(ascending=False))

2026-01-06 00:00:00 2026-09-25 00:00:00
31
     File Date                                 Company  Shares_offered_value
56  2026-09-01  Hornbeck Offshore Services (Withdrawn)               100.000
376 2026-03-06          Invea Therapeutics (Withdrawn)                75.000
132 2026-07-31               Nuclea Energy (Withdrawn)                50.400
360 2026-03-16                       Firefly Automatix                25.025
90  2026-08-18           Living Homeopathy (Withdrawn)                18.750
272 2026-05-07                AiXin Life International                 9.520
264 2026-05-13                   Ultratrex (Withdrawn)                 6.250
466 2026-01-27                                 SFIDA X                 5.500
Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             197.2500
Technologies        184.9000
Group                32.5000
Name: Shares_offered_value, dtype: float64


# Question 2:

In [7]:
import io, requests
import pandas as pd
import numpy as np
import yfinance as yf

url = "https://www.iposcoop.com/2025-pricings/"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
ipo = pd.read_html(io.StringIO(resp.text))[0]
print(ipo.columns.tolist())
print(len(ipo))  # about 231
print(ipo.head())

['Company', 'Symbol', 'Industry', 'Offer Date', 'Shares (millions)', 'Offer Price', '1st Day Close', 'Current Price', 'Return', 'SCOOP Rating']
231
                                 Company Symbol           Industry  \
0  Vine Hill Capital Investment Corp. II  VHCPU        Blank Check   
1                         Andersen Group   ANDG  Consumer Services   
2                           Medline Inc.   MDLN        Health Care   
3                      Wealthfront Corp.   WLTH         Financials   
4                Lumexa Imaging Holdings   LMRI        Health Care   

   Offer Date  Shares (millions) Offer Price 1st Day Close Current Price  \
0  12/18/2025               20.0      $10.00         $0.00         $0.00   
1  12/17/2025               11.0      $16.00        $23.50        $23.79   
2  12/17/2025              216.0      $29.00        $41.00        $43.83   
3  12/12/2025               34.6      $14.00        $14.19         $8.49   
4  12/11/2025               25.0      $18.50       

In [10]:
ipo["Offer Date"] = pd.to_datetime(ipo["Offer Date"], errors="coerce")
ipo = ipo[ipo["Offer Date"] < "2025-09-01"]

# find your return column name from the printout, e.g. "% Change" or "Return"
ret_col = "Return"
ipo["ret"] = pd.to_numeric(
    ipo[ret_col].astype(str).str.replace("%", "").str.replace(",", ""),
    errors="coerce"
)
ipo = ipo[ipo["ret"] != 0]
print(len(ipo))  # about 148

tickers = ipo["Symbol"].astype(str).str.strip().unique().tolist()

146


In [11]:
frames = []
for t in tickers:
    try:
        d = yf.download(t, start="2025-01-01", progress=False)
    except Exception:
        continue
    if d is None or d.empty:
        continue
    d.columns = d.columns.get_level_values(0)  # MultiIndex fix
    d["Ticker"] = t
    frames.append(d.reset_index())

stocks_df = pd.concat(frames, ignore_index=True)
print(stocks_df["Ticker"].nunique())  # about 134

/tmp/ipykernel_1189/2465564449.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, start="2025-01-01", progress=False)
/tmp/ipykernel_1189/2465564449.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, start="2025-01-01", progress=False)
/tmp/ipykernel_1189/2465564449.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, start="2025-01-01", progress=False)
/tmp/ipykernel_1189/2465564449.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, start="2025-01-01", progress=False)
/tmp/ipykernel_1189/2465564449.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, start="2025-01-01", progress=False)
/tmp/ipykernel_1189/2465564449.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d = yf.download(t, star

132


In [12]:
stocks_df = stocks_df.sort_values(["Ticker", "Date"])
g = stocks_df.groupby("Ticker")["Close"]

stocks_df["growth_252d"] = stocks_df["Close"] / g.shift(252)
stocks_df["volatility"] = g.transform(lambda s: s.rolling(30).std()) * np.sqrt(252)
stocks_df["Sharpe"] = (stocks_df["growth_252d"] - 0.05) / stocks_df["volatility"]

In [13]:
day = stocks_df[stocks_df["Date"] == "2026-09-11"]
print(day[["growth_252d", "volatility", "Sharpe"]].describe())

Price  growth_252d  volatility      Sharpe
count   129.000000  130.000000  129.000000
mean      1.065215   27.725051         inf
std       3.059657   67.095064         NaN
min       0.001005    0.000000   -0.040147
25%       0.149038    2.325895    0.011942
50%       0.596026    7.927896    0.048009
75%       1.040233   23.659042    0.134248
max      33.638270  627.698106         inf


# Question 3:

In [14]:
g = stocks_df.groupby("Ticker")["Close"]

for m in range(1, 13):
    stocks_df[f"future_growth_{m}_m"] = g.shift(-21 * m) / stocks_df["Close"]

# first trading day per ticker
min_dates = (
    stocks_df.groupby("Ticker")["Date"].min()
    .reset_index()
    .rename(columns={"Date": "min_date"})
)

# inner join keeps only each ticker's IPO-day row
entry = stocks_df.merge(
    min_dates,
    left_on=["Ticker", "Date"],
    right_on=["Ticker", "min_date"],
    how="inner",
)

cols = [f"future_growth_{m}_m" for m in range(1, 13)]
stats = entry[cols].describe()
print(stats)
print(stats.loc["50%"].idxmax(), stats.loc["50%"].max())

       future_growth_1_m  future_growth_2_m  future_growth_3_m  \
count         131.000000         130.000000         130.000000   
mean           96.475452          68.494523          51.642359   
std          1092.037636         767.322855         575.469311   
min             0.072941           0.106197           0.090147   
25%             0.709910           0.582405           0.449793   
50%             0.930622           0.891684           0.827951   
75%             1.114924           1.171005           1.139365   
max         12500.000279        8750.000196        6562.500147   

       future_growth_4_m  future_growth_5_m  future_growth_6_m  \
count         130.000000         130.000000         130.000000   
mean           22.872042          54.660308          64.504145   
std           248.776445         611.660280         723.482269   
min             0.063927           0.032245           0.022075   
25%             0.404543           0.431361           0.315944   
50%      

# Question 4:

In [15]:
import gdown
import pandas as pd

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")

print(df.columns.tolist())  # check the exact RSI and date column names

Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=a87b63c8-6ae0-4a57-b7cf-369316310cca
To: /content/data.parquet
100%|██████████| 130M/130M [00:02<00:00, 62.8MB/s]


['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits', 'Ticker', 'Year', 'Month', 'Weekday', 'Date', 'growth_1d', 'growth_3d', 'growth_7d', 'growth_30d', 'growth_90d', 'growth_365d', 'growth_future_30d', 'SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative', 'volatility', 'is_positive_growth_30d_future', 'ticker_type', 'index_x', 'adx', 'adxr', 'apo', 'aroon_1', 'aroon_2', 'aroonosc', 'bop', 'cci', 'cmo', 'dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext', 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix', 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo', 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk', 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr', 'index_y', 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase', 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine', 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice', 'index', 'cdl2crows', '

In [16]:
df["Date"] = pd.to_datetime(df["Date"])

selected_df = df[
    (df["rsi"] < 30)
    & (df["Date"] >= "2000-01-01")
    & (df["Date"] <= "2025-06-01")
]

print(len(selected_df))  # homework says about 5,206

net_income = 1000 * (selected_df["growth_future_30d"] - 1).sum()
print(net_income, net_income / 1000)

5206
65805.58960747933 65.80558960747933
